<a href="https://colab.research.google.com/github/azrasm/waste-classification/blob/main/reniranje_finalnog_modela.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

ponovno treniranje modela sa class_weights

In [ ]:
import tensorflow as tf
from keras import layers, models, optimizers, callbacks, applications
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import os
import itertools
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
train_dir = "/content/drive/MyDrive/POOS projekat/PROCESSED_DATASET/TRAIN"
val_dir   = "/content/drive/MyDrive/POOS projekat/PROCESSED_DATASET/VAL"
test_dir  = "/content/drive/MyDrive/POOS projekat/PROCESSED_DATASET/TEST"

In [ ]:
batch_size = 16

from sklearn.utils import class_weight

train_labels_all = []
for f in os.listdir(train_dir):
    if f.endswith(".npz"):
        data = np.load(os.path.join(train_dir, f))
        train_labels_all.extend(data["y"])
train_labels_all = np.array(train_labels_all)

weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels_all),
    y=train_labels_all
)
class_weights_dict = dict(enumerate(weights))
print("Class weights:", class_weights_dict)


def batch_generator_with_weights(batch_dir, batch_size=16):
    batch_files = sorted(f for f in os.listdir(batch_dir) if f.endswith(".npz"))

    while True:
        X_batch, y_batch, w_batch = [], [], []
        for bf in batch_files:
            data = np.load(os.path.join(batch_dir, bf))
            X, y = data["X"], data["y"]
            for i in range(len(X)):
                X_batch.append(X[i])
                y_batch.append(y[i])
                w_batch.append(class_weights_dict[y[i]])  # dodaj težinu

                if len(X_batch) == batch_size:
                    yield np.array(X_batch), np.array(y_batch), np.array(w_batch)
                    X_batch, y_batch, w_batch = [], [], []

Class weights: {0: np.float64(1.0), 1: np.float64(1.0)}


In [ ]:
steps_per_epoch = sum(len(np.load(os.path.join(train_dir, f))["X"]) for f in os.listdir(train_dir)) // batch_size
validation_steps = sum(len(np.load(os.path.join(val_dir, f))["X"]) for f in os.listdir(val_dir)) // batch_size

In [ ]:
from keras.applications import ResNet50
from keras.layers import Dense, GlobalAveragePooling2D, Dropout
from keras.models import Model
from keras.optimizers import Adam

base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

for layer in base_model.layers:
    layer.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.5)(x)
output = Dense(1, activation="sigmoid")(x)

model = Model(inputs=base_model.input, outputs=output)

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ input_layer[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_3_c

 Total params: 24,112,513 (91.98 MB)

 Trainable params: 524,801 (2.00 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [ ]:
from keras.callbacks import EarlyStopping, ModelCheckpoint

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    "/content/drive/MyDrive/POOS projekat/resnet50_waste_classifier_best.keras",
    monitor='val_loss',
    save_best_only=True
)

In [ ]:
train_gen = batch_generator_with_weights(train_dir, batch_size)
val_gen   = batch_generator_with_weights(val_dir, batch_size)

history = model.fit(
    train_gen,
    steps_per_epoch=steps_per_epoch,
    validation_data=val_gen,
    validation_steps=validation_steps,
    epochs=30,
    callbacks=[early_stop, checkpoint]
)

Epoch 1/30
1288/1288 ━━━━━━━━━━━━━━━━━━━━ 136s 96ms/step - accuracy: 0.6063 - loss: 0.6709 - val_accuracy: 0.7077 - val_loss: 0.5914
Epoch 2/30
1288/1288 ━━━━━━━━━━━━━━━━━━━━ 140s 109ms/step - accuracy: 0.6781 - loss: 0.6022 - val_accuracy: 0.7137 - val_loss: 0.5817
Epoch 3/30
1288/1288 ━━━━━━━━━━━━━━━━━━━━ 118s 92ms/step - accuracy: 0.6884 - loss: 0.5916 - val_accuracy: 0.7212 - val_loss: 0.5766
Epoch 4/30
1288/1288 ━━━━━━━━━━━━━━━━━━━━ 118s 92ms/step - accuracy: 0.7004 - loss: 0.5807 - val_accuracy: 0.7172 - val_loss: 0.5770
Epoch 5/30
1288/1288 ━━━━━━━━━━━━━━━━━━━━ 118s 92ms/step - accuracy: 0.7046 - loss: 0.5745 - val_accuracy: 0.7185 - val_loss: 0.5768
Epoch 6/30
1288/1288 ━━━━━━━━━━━━━━━━━━━━ 117s 91ms/step - accuracy: 0.7112 - loss: 0.5677 - val_accuracy: 0.7232 - val_loss: 0.5724
Epoch 7/30
1288/1288 ━━━━━━━━━━━━━━━━━━━━ 116s 90ms/step - accuracy: 0.7143 - loss: 0.5634 - val_accuracy: 0.7181 - val_loss: 0.5724
Epoch 8/30
1288/1288 ━━━━━━━━━━━━━━━━━━━━ 116s 90ms/step - accuracy:

In [ ]:
model.save("/content/drive/MyDrive/POOS projekat/resnet50_waste_classifier_best.keras")